# Phase 2 — Exploratory Data Analysis (EDA)
## YouTube Quality Analyzer — Corpus de commentaires

**Objectif** : Explorer et valider le dataset CSV avant de lancer le pipeline multi-agents.

**Livrables** :
- Statistiques descriptives du corpus
- Distribution des longueurs et des langues
- Métriques d'engagement (author_likes, reply_count)
- Flags de bruit heuristiques
- Export vers `data/clean/`

**Tag Git cible** : `v0.1.0`

---
## 0. Configuration

In [ ]:
import random, re, html as html_lib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

CSV_PATH     = '../data/raw/comments_raw.csv'
CLEAN_OUTPUT = '../data/clean/comments_eda_preview.csv'

REQUIRED_COLS = ['text']
OPTIONAL_COLS = ['video_id', 'author_likes', 'reply_count']

COLUMN_ALIASES = {
    'texte_commentaire':    'text',
    'comment_text':         'text',
    'body':                 'text',
    'nb_likes_commentaire': 'author_likes',
    'likes':                'author_likes',
    'nb_reponses':          'reply_count',
    'replies':              'reply_count',
    'commentaire_id':       'comment_id',
    'publie_le':            'published_at',
}

print('Configuration OK — seed =', SEED)
print('CSV source :', CSV_PATH)

---
## 1. Chargement et validation du CSV

In [ ]:
df = pd.read_csv(CSV_PATH)
print('Colonnes brutes du CSV :', list(df.columns))

# Renommage automatique des colonnes (meme logique que A1 Loader)
rename_map = {col: COLUMN_ALIASES[col] for col in df.columns if col in COLUMN_ALIASES}
if rename_map:
    df = df.rename(columns=rename_map)
    print(f'Colonnes renommees : {rename_map}')

print('Colonnes apres renommage :', list(df.columns))

# Decodage HTML (&#39; -> ', <br> -> espace) -- present sur ~32% du corpus YouTube
df['text'] = df['text'].astype(str).apply(
    lambda t: re.sub(r'<[^>]+>', ' ', html_lib.unescape(t))
)

print(f'Shape : {df.shape[0]:,} lignes x {df.shape[1]} colonnes')
df.head()

In [ ]:
missing_required = [c for c in REQUIRED_COLS if c not in df.columns]
if missing_required:
    raise ValueError(
        f'Colonnes requises manquantes : {missing_required}\n'
        f'Colonnes disponibles : {list(df.columns)}'
    )

present_optional = [c for c in OPTIONAL_COLS if c in df.columns]
missing_optional  = [c for c in OPTIONAL_COLS if c not in df.columns]
print(f'OK Colonnes requises       : {REQUIRED_COLS}')
print(f'OK Colonnes opt. presentes : {present_optional}')
if missing_optional:
    print(f'   Colonnes opt. absentes  : {missing_optional}')

print(f'\nValeurs nulles :')
print(df[REQUIRED_COLS + present_optional].isnull().sum().to_string())
print(f'\nDoublons exacts (texte) : {df["text"].duplicated().sum():,}')

In [ ]:
print('Types :')
print(df.dtypes.to_string())

---
## 2. Statistiques descriptives

In [ ]:
df_clean = df.copy()
df_clean['text'] = df_clean['text'].astype(str).str.strip()
df_clean = df_clean[df_clean['text'].str.len() > 0].copy()

df_clean['char_length'] = df_clean['text'].str.len()
df_clean['word_count']  = df_clean['text'].str.split().str.len()

print(f'Apres suppression vides : {len(df_clean):,} (supprimes : {len(df) - len(df_clean)})')
print()
print(df_clean[['char_length', 'word_count']].describe().round(2).to_string())

In [ ]:
single_word = df_clean[df_clean['word_count'] == 1]
print(f'Commentaires 1 mot : {len(single_word):,} ({100*len(single_word)/len(df_clean):.1f}%)')
print(single_word['text'].value_counts().head(10).to_string())

long_comments = df_clean[df_clean['word_count'] > 100]
print(f'\nCommentaires >100 mots : {len(long_comments):,} ({100*len(long_comments)/len(df_clean):.1f}%)')

---
## 3. Distribution des longueurs

In [ ]:
import pathlib; pathlib.Path('../docs/phase2_data').mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df_clean['char_length'].clip(upper=500), bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Longueur en caracteres (cap 500)')
axes[0].set_xlabel('Caracteres')
axes[0].set_ylabel('Nb commentaires')
axes[0].axvline(3, color='red', linestyle='--', label='Min A2 (3 chars)')
axes[0].legend(fontsize=8)

axes[1].hist(df_clean['word_count'].clip(upper=100), bins=50, color='teal', edgecolor='white')
axes[1].set_title('Nombre de mots (cap 100)')
axes[1].set_xlabel('Mots')
axes[1].set_ylabel('Nb commentaires')
axes[1].axvline(100, color='red', linestyle='--', label='Seuil long (Musleh 2023)')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('../docs/phase2_data/fig_length_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Sauvegarde : docs/phase2_data/fig_length_distribution.png')

---
## 4. Détection des langues

In [ ]:
def safe_detect(text):
    try:
        from langdetect import detect
        return detect(str(text))
    except Exception:
        return 'unknown'

sample_size = min(500, len(df_clean))
df_sample = df_clean.sample(sample_size, random_state=SEED).copy()
print(f'Detection de langue sur {sample_size} commentaires...')
df_sample['language'] = df_sample['text'].apply(safe_detect)
lang_counts = df_sample['language'].value_counts()
print(lang_counts.head(10).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
top_langs = lang_counts.head(10)
bars = ax.barh(top_langs.index[::-1], top_langs.values[::-1], color='steelblue')
ax.set_title('Top 10 langues detectees (500 commentaires)')
ax.set_xlabel('Nb commentaires')
for bar, val in zip(bars, top_langs.values[::-1]):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val} ({100*val/sample_size:.1f}%)', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('../docs/phase2_data/fig_language_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Métriques d'engagement

In [ ]:
engagement_cols = [c for c in ['author_likes', 'reply_count'] if c in df_clean.columns]

if engagement_cols:
    print(df_clean[engagement_cols].describe().round(2).to_string())

    fig, axes = plt.subplots(1, len(engagement_cols), figsize=(6*len(engagement_cols), 4))
    if len(engagement_cols) == 1:
        axes = [axes]

    for ax, col in zip(axes, engagement_cols):
        cap = df_clean[col].quantile(0.95)
        ax.hist(df_clean[col].dropna().clip(upper=cap), bins=40, color='coral', edgecolor='white')
        ax.set_title(f'{col} (cap 95e percentile)')
        ax.set_xlabel(col)
        ax.set_ylabel('Nb commentaires')
        med = df_clean[col].median()
        ax.axvline(med, color='navy', linestyle='--', label=f'Mediane: {med:.0f}')
        ax.legend(fontsize=9)

    plt.tight_layout()
    plt.savefig('../docs/phase2_data/fig_engagement_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Warning : author_likes et reply_count absents du CSV.')

---
## 6. Analyse par vidéo

In [ ]:
if 'video_id' in df_clean.columns:
    video_counts = df_clean['video_id'].value_counts()
    print(f'Videos distinctes : {df_clean["video_id"].nunique():,}')
    print(f'Commentaires/video — moy : {video_counts.mean():.0f}, med : {video_counts.median():.0f}')
    print(video_counts.head(10).to_frame('nb_commentaires').to_string())

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(video_counts, bins=30, color='mediumpurple', edgecolor='white')
    ax.set_title('Distribution du nb de commentaires par video')
    ax.set_xlabel('Nb commentaires')
    ax.set_ylabel('Nb videos')
    plt.tight_layout()
    plt.savefig('../docs/phase2_data/fig_comments_per_video.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Warning : colonne video_id absente.')

---
## 7. Heuristiques de bruit pré-pipeline

Ces flags sont des estimations rapides **avant** le passage par A5 (Noise Detector LLM).

In [ ]:
def flag_noise(text):
    text = str(text).strip()
    if len(text) < 3: return 'trop_court'
    if re.fullmatch(r'[^\w\s]+|[\d\W]+', text): return 'emoji_only'
    if len(text.split()) == 1: return 'reaction_vide'
    if re.search(r'https?://\S+|www\.\S+|bit\.ly/\S+', text, re.IGNORECASE): return 'contient_url'
    if re.search(r'(.)\1{4,}', text): return 'lettres_repetees'
    spam_kw = r'\b(subscribe|abonne|abonnez|follow|check.?out|click.?here|gagne[rz]?|earn|bit\.ly)\b'
    if re.search(spam_kw, text, re.IGNORECASE): return 'spam'
    return 'ok'

df_clean['noise_flag'] = df_clean['text'].apply(flag_noise)
noise_dist = df_clean['noise_flag'].value_counts()
print(noise_dist.to_string())
ratio_bruit = 100 * (1 - noise_dist.get('ok', 0) / len(df_clean))
print(f'\nRatio de bruit estime (heuristique) : {ratio_bruit:.1f}%')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#2ecc71' if k == 'ok' else '#e74c3c' for k in noise_dist.index]
bars = ax.bar(noise_dist.index, noise_dist.values, color=colors, edgecolor='white')
ax.set_title('Flags de bruit heuristique (pre-pipeline A5)')
ax.set_ylabel('Nb commentaires')
for bar, val in zip(bars, noise_dist.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
            f'{val} ({100*val/len(df_clean):.1f}%)', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig('../docs/phase2_data/fig_noise_flags.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Exemples de commentaires

In [ ]:
clean_sample = df_clean[df_clean['noise_flag'] == 'ok'].copy()
print(f'Commentaires propres : {len(clean_sample):,}')

if 'author_likes' in clean_sample.columns:
    print('\nTop 5 les plus aimes :')
    print(clean_sample.nlargest(5, 'author_likes')[['text', 'author_likes']].to_string())
else:
    print('\n5 exemples aleatoires :')
    print(clean_sample.sample(5, random_state=SEED)[['text']].to_string())

In [ ]:
noisy_sample = df_clean[df_clean['noise_flag'] != 'ok']
if len(noisy_sample) > 0:
    print(f'Exemples de commentaires bruites ({len(noisy_sample):,} total) :')
    print(noisy_sample.groupby('noise_flag').head(2)[['noise_flag', 'text']].to_string())
else:
    print('Aucun commentaire noisy detecte.')

---
## 9. Résumé et export

In [ ]:
n_total  = len(df)
n_clean  = len(df_clean)
n_ok     = len(df_clean[df_clean['noise_flag'] == 'ok'])
n_noisy  = n_clean - n_ok
n_dup    = df['text'].duplicated().sum()
n_videos = df_clean['video_id'].nunique() if 'video_id' in df_clean.columns else 'N/A'

print('=' * 52)
print('RESUME EDA — CORPUS DE COMMENTAIRES')
print('=' * 52)
print(f'Total brut               : {n_total:>8,}')
print(f'Doublons exacts          : {n_dup:>8,}  ({100*n_dup/n_total:.1f}%)')
print(f'Apres filtrage A2        : {n_clean:>8,}')
print(f'Propres (heuristique)    : {n_ok:>8,}  ({100*n_ok/n_clean:.1f}%)')
print(f'Bruites (heuristique)    : {n_noisy:>8,}  ({100*n_noisy/n_clean:.1f}%)')
print(f'Videos distinctes        : {str(n_videos):>8}')
print(f'Longueur moy (chars)     : {df_clean["char_length"].mean():>8.0f}')
print(f'Longueur med (mots)      : {df_clean["word_count"].median():>8.0f}')
print('=' * 52)

In [ ]:
import pathlib
pathlib.Path('../data/clean').mkdir(parents=True, exist_ok=True)

export_cols = ['text', 'noise_flag', 'char_length', 'word_count']
for opt in ['video_id', 'author_likes', 'reply_count']:
    if opt in df_clean.columns:
        export_cols = [opt] + export_cols if opt == 'video_id' else export_cols + [opt]

df_export = df_clean[[c for c in export_cols if c in df_clean.columns]]
df_export.to_csv(CLEAN_OUTPUT, index=False, encoding='utf-8')
print(f'Export : {CLEAN_OUTPUT} ({len(df_export):,} lignes)')

---
## 10. Gold Standard — Annotation automatique

| Étape | Méthode |
|---|---|
| Sélection stratifiée | 100 commentaires (strates : bruit × longueur) |
| Annotation | 2 passages LLM indépendants |
| Kappa | Cohen's kappa — seuil 0.7 (AC-08 PRD) |
| Export | `data/gold_standard/gold_standard.jsonl` |
| Tag Git | `v0.1.0` si kappa > 0.7 |

In [ ]:
import subprocess, sys

result = subprocess.run(
    [
        sys.executable,
        '../scripts/annotate_gold_standard.py',
        '--csv',       CSV_PATH,
        '--n_samples', '100',
        '--output',    '../data/gold_standard/gold_standard.jsonl',
        # '--auto_tag',  # decommenter pour creer le tag Git v0.1.0
    ],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[:800])